# Bronze Layer Ingestion — REST API Source

**Lakehouse:** `bronze_citadel_mga`  
**Owner:** Venkata Pothireddy  
**Purpose:** Ingest three entities (policies, claims, agents) from a REST API into Delta tables in the bronze layer, with full ingestion lineage metadata.

This notebook is the API-source counterpart to `bronze_ingest_all` (CSV-source). Both notebooks land data in the same lakehouse but in separate tables, allowing the silver layer to union them with full source attribution.

## Source

A mock IMS (Insurance Management System) API exposing three endpoints:
- `GET /policies` — policy records with pagination
- `GET /claims` — claim records with pagination  
- `GET /agents` — agent/producer records with pagination

The API supports incremental pulls via the `updated_since` query parameter, though this notebook performs a full pull for prototype simplicity.

In production, this would be a real Vertafore IMS API. For this prototype, a FastAPI mock running on localhost is exposed through ngrok so Fabric can reach it.

## Outputs

Three Delta tables in `Tables/`:

| Table | Source endpoint |
|---|---|
| `bronze_policies_api` | `GET /policies` |
| `bronze_claims_api` | `GET /claims` |
| `bronze_agents_api` | `GET /agents` |

Each row carries four lineage columns:

| Column | Purpose |
|---|---|
| `_ingestion_timestamp` | When this row was loaded into bronze |
| `_source_endpoint` | Which API endpoint the row came from |
| `_batch_id` | Which notebook run loaded this batch |
| `_ingestion_method` | Source type — `"api"` for all rows from this notebook |

## Design principles

- **No transformations.** Bronze preserves source data as received.
- **Idempotent.** Each run overwrites the entire table — bronze rebuilds cleanly.
- **Traceable.** Every row is attributable to a specific endpoint and batch.
- **Schema enforcement on.** Schema changes require explicit `ALTER TABLE`, not implicit write options.
- **Paginated.** Walks through API results page by page until exhausted.


In [1]:
# Cell 2 — Imports and config
#
# requests:  industry-standard Python HTTP client. We use it to call the
#            REST API and parse JSON responses.
# pyspark.sql.functions: lit() for constant-value columns, current_timestamp()
#            for the ingestion timestamp.
# uuid + datetime: generate a unique batch ID, same pattern as the CSV notebook.

import requests
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType
import uuid
from datetime import datetime

# Generate a unique batch ID for this run.
# Format: bronze_api_YYYYMMDD_HHMMSS_<short-uuid>
#
# Same convention as the CSV notebook's batch ID, but prefixed with "api"
# so we can distinguish source pipelines when auditing _batch_id values
# across both bronze tables.

BATCH_ID = f"bronze_api_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
print(f"Batch ID for this run: {BATCH_ID}")

# Base URL of the source API.
#
# In production this would be the Vertafore IMS API endpoint.
# For this prototype it's the FastAPI mock running on localhost,
# exposed through ngrok so Fabric (running in Microsoft's cloud)
# can reach it.
#
# NOTE: ngrok URLs change every time the tunnel restarts on the free tier.
# If this notebook fails with a connection error, update the URL below
# with the current forwarding URL shown in your ngrok terminal.

API_BASE_URL = "https://eclair-removable-essential.ngrok-free.dev"

# Pagination batch size — how many records to request per API call.
# The API supports up to 1000 per call (its MAX_LIMIT). We use 500 as a
# reasonable middle ground — large enough to minimize HTTP round-trips,
# small enough to keep memory usage bounded.

PAGE_SIZE = 500

# Default headers for every request.
#
# ngrok-skip-browser-warning bypasses ngrok's HTML interstitial page that
# normally appears on browser visits. Programmatic clients should send this
# header so they get raw API responses instead of the warning HTML.
# Real APIs in production won't need this — it's only for the ngrok tunnel.
#
# User-Agent identifies the calling client. Convention to set a meaningful
# one rather than leaving it as Python's default.

DEFAULT_HEADERS = {
    "ngrok-skip-browser-warning": "true",
    "User-Agent": "citadel-fabric-bronze-ingestion/1.0",
}

print(f"API base URL: {API_BASE_URL}")
print(f"Page size:    {PAGE_SIZE}")

StatementMeta(, 74e621a2-8cca-4884-8a82-e16c13388bb2, 3, Finished, Available, Finished, False)

Batch ID for this run: bronze_api_20260530_135132_b49ae0f9
API base URL: https://eclair-removable-essential.ngrok-free.dev
Page size:    500


## Ingestion Function

We define one reusable function `ingest_api_to_delta()` that handles a single API endpoint → Delta conversion. All three endpoints (policies, claims, agents) use the same function.

**What the function does:**

1. Call the API endpoint in a paginated loop (offset/limit) until all records are retrieved
2. Accumulate the records in memory
3. Convert the JSON records to a Spark DataFrame
4. Add four lineage columns (`_ingestion_timestamp`, `_source_endpoint`, `_batch_id`, `_ingestion_method`)
5. Write to `Tables/` as a Delta table in overwrite mode

**Pagination strategy:** the API exposes `limit` and `offset` parameters and returns a `pagination.has_more` flag in each response. We keep requesting pages until `has_more` is `false`.

**Error handling:** any non-200 HTTP response raises an exception, halting ingestion before partial data lands in the table. Bronze is rebuildable, so failing fast is preferable to silent partial writes.

**Overwrite mode rationale:** same as the CSV notebook. Bronze is fully rebuildable from source. Each run produces a clean table.


In [3]:
# Cell 4 — Reusable API ingestion function

def fetch_all_pages(endpoint_path: str) -> list:
    """
    Walk through a paginated API endpoint and return all records as a list.
    
    Uses offset-based pagination: starts at offset=0, fetches a page,
    increments the offset by PAGE_SIZE, repeats until the API reports
    no more data via pagination.has_more = false.
    
    Args:
        endpoint_path: API path starting with "/", e.g. "/policies"
    
    Returns:
        A list of dicts, one per record across all pages.
    
    Raises:
        requests.HTTPError: if any page request returns non-2xx.
    """
    all_records = []
    offset = 0
    page_number = 1
    
    while True:
        url = f"{API_BASE_URL}{endpoint_path}"
        params = {"limit": PAGE_SIZE, "offset": offset}
        
        print(f"  Page {page_number}: GET {endpoint_path}?limit={PAGE_SIZE}&offset={offset}")
        
        # requests.get returns a Response object.
        # .raise_for_status() throws an exception if the status code
        # is 4xx or 5xx — preferable to silently accumulating bad data.
        response = requests.get(url, params=params, headers=DEFAULT_HEADERS, timeout=30)
        response.raise_for_status()
        
        payload = response.json()
        
        # The API wraps results in a standard envelope:
        # { "data": [...], "pagination": { "limit", "offset", "total", "has_more" } }
        records = payload["data"]
        pagination = payload["pagination"]
        
        all_records.extend(records)
        
        # Stop if the API tells us there are no more pages.
        if not pagination["has_more"]:
            print(f"  Done. {len(all_records):,} records total across {page_number} page(s).")
            break
        
        offset += PAGE_SIZE
        page_number += 1
    
    return all_records


def ingest_api_to_delta(endpoint_path: str, target_table_name: str) -> int:
    """
    Fetch all records from a paginated API endpoint, convert to a Spark
    DataFrame, add bronze lineage columns, and write to a Delta table.
    
    Args:
        endpoint_path: API path, e.g. "/policies"
        target_table_name: name of the target Delta table, e.g. "bronze_policies_api"
    
    Returns:
        Row count written (sanity check).
    """
    print(f"Fetching {endpoint_path}...")
    
    # Step 1: fetch all records via paginated calls.
    records = fetch_all_pages(endpoint_path)
    
    if not records:
        print(f"  WARNING: no records returned from {endpoint_path}")
        return 0
    
    # Step 2: convert the list of dicts into a Spark DataFrame.
    #
    # spark.createDataFrame() can take a Python list of dicts directly.
    # Spark infers the schema from the first row's keys and types.
    # For bronze we accept whatever the API gives us — silver re-types.
    df = spark.createDataFrame(records)
    
    # Step 3: add the four bronze lineage columns.
    df_with_lineage = (
        df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_endpoint", lit(endpoint_path))
        .withColumn("_batch_id", lit(BATCH_ID))
        .withColumn("_ingestion_method", lit("api"))
    )
    
    # Step 4: write to the lakehouse as a managed Delta table.
    # Schema enforcement is on — schema changes require explicit ALTER TABLE.
    (
        df_with_lineage
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table_name)
    )
    
    row_count = df_with_lineage.count()
    print(f"  → wrote {row_count:,} rows to table {target_table_name}")
    return row_count

StatementMeta(, 74e621a2-8cca-4884-8a82-e16c13388bb2, 5, Finished, Available, Finished, False)

## Run Ingestion for All Three Endpoints

Call `ingest_api_to_delta()` once per endpoint. Order doesn't matter at the bronze layer because no referential integrity is enforced here — that's silver's job.

The three target tables (`bronze_policies_api`, `bronze_claims_api`, `bronze_agents_api`) are distinct from the CSV-source tables (`bronze_policies`, etc.). The silver layer reconciles both sources.

Row counts collected as we go for verification in the next cell.

In [5]:
# Cell 6 — Ingest all three API endpoints into Delta tables
#
# Convention: bronze_<entity>_api — the "_api" suffix distinguishes
# these tables from the CSV-sourced tables (bronze_policies, etc.)
# in the same lakehouse.

ingestion_plan = [
    ("/policies", "bronze_policies_api"),
    ("/claims",   "bronze_claims_api"),
    ("/agents",   "bronze_agents_api"),
]

# Run ingestion for each endpoint. Collect row counts.
row_counts = {}

for endpoint_path, target_table_name in ingestion_plan:
    row_counts[target_table_name] = ingest_api_to_delta(
        endpoint_path=endpoint_path,
        target_table_name=target_table_name,
    )
    print()  # blank line between endpoints for readability

print("=" * 50)
print("API ingestion complete. Summary:")
print("=" * 50)
for table_name, count in row_counts.items():
    print(f"  {table_name:30s}  {count:>8,} rows")

StatementMeta(, 74e621a2-8cca-4884-8a82-e16c13388bb2, 7, Finished, Available, Finished, False)

Fetching /policies...
  Page 1: GET /policies?limit=500&offset=0
  Page 2: GET /policies?limit=500&offset=500
  Page 3: GET /policies?limit=500&offset=1000
  Page 4: GET /policies?limit=500&offset=1500
  Page 5: GET /policies?limit=500&offset=2000
  Page 6: GET /policies?limit=500&offset=2500
  Page 7: GET /policies?limit=500&offset=3000
  Page 8: GET /policies?limit=500&offset=3500
  Page 9: GET /policies?limit=500&offset=4000
  Page 10: GET /policies?limit=500&offset=4500
  Page 11: GET /policies?limit=500&offset=5000
  Page 12: GET /policies?limit=500&offset=5500
  Page 13: GET /policies?limit=500&offset=6000
  Page 14: GET /policies?limit=500&offset=6500
  Page 15: GET /policies?limit=500&offset=7000
  Page 16: GET /policies?limit=500&offset=7500
  Page 17: GET /policies?limit=500&offset=8000
  Page 18: GET /policies?limit=500&offset=8500
  Page 19: GET /policies?limit=500&offset=9000
  Page 20: GET /policies?limit=500&offset=9500
  Done. 10,000 records total across 20 page(s).
  →

## Verification

Sanity-check the ingestion by querying each Delta table and confirming:
1. Row counts match what the ingestion summary printed
2. The four lineage columns are present and populated
3. Every row has `_ingestion_method = "api"`
4. A sample of rows looks reasonable

This mirrors the verification pattern in the CSV bronze notebook — same checks, different source.

In [6]:
# Cell 8 — Verify all three API bronze tables

api_bronze_tables = [
    "bronze_policies_api",
    "bronze_claims_api",
    "bronze_agents_api",
]

print("Table verification:")
print("=" * 75)

for table_name in api_bronze_tables:
    df = spark.sql(f"SELECT * FROM {table_name}")
    row_count = df.count()
    col_count = len(df.columns)
    
    # Confirm the four lineage columns are present
    required_lineage = [
        "_ingestion_timestamp",
        "_source_endpoint",
        "_batch_id",
        "_ingestion_method",
    ]
    has_lineage = all(col in df.columns for col in required_lineage)
    lineage_marker = "✓" if has_lineage else "✗"
    
    # Confirm _ingestion_method is "api" for every row.
    # If even one row says something else, something went wrong with lineage tagging.
    method_check = spark.sql(f"""
        SELECT DISTINCT _ingestion_method
        FROM {table_name}
    """).collect()
    methods = [row["_ingestion_method"] for row in method_check]
    method_marker = "✓" if methods == ["api"] else f"✗ ({methods})"
    
    print(f"  {table_name:25s}  {row_count:>8,} rows  {col_count:>3} cols  "
          f"lineage: {lineage_marker}  method: {method_marker}")

# Show a sample row from bronze_policies_api so we can see what an API-sourced
# bronze record looks like, including lineage columns.
print("\nSample rows from bronze_policies_api:")
spark.sql("""
    SELECT
        policy_id,
        program_code,
        premium,
        status,
        _source_endpoint,
        _ingestion_method,
        _batch_id
    FROM bronze_policies_api
    LIMIT 5
""").show(truncate=False)

StatementMeta(, 74e621a2-8cca-4884-8a82-e16c13388bb2, 8, Finished, Available, Finished, False)

Table verification:
  bronze_policies_api          10,000 rows   14 cols  lineage: ✓  method: ✓
  bronze_claims_api             2,000 rows   11 cols  lineage: ✓  method: ✓
  bronze_agents_api               200 rows   12 cols  lineage: ✓  method: ✓

Sample rows from bronze_policies_api:
+----------+------------+--------+---------+----------------+-----------------+-----------------------------------+
|policy_id |program_code|premium |status   |_source_endpoint|_ingestion_method|_batch_id                          |
+----------+------------+--------+---------+----------------+-----------------+-----------------------------------+
|POL-001025|CONT        |7351.26 |EXPIRED  |/policies       |api              |bronze_api_20260530_135132_b49ae0f9|
|POL-001026|ENRG        |30000.0 |CANCELLED|/policies       |api              |bronze_api_20260530_135132_b49ae0f9|
|POL-001027|CONT        |5000.0  |ACTIVE   |/policies       |api              |bronze_api_20260530_135132_b49ae0f9|
|POL-001028|LIFE 